# Predicting a warranty claim within a year
Protocol: `protocol.yaml`. Holdout rows from `holdout.csv`, 5-fold stratified CV on training rows, choose by mean CV ROC AUC, score the holdout once.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
for f, h in [('data.csv', '25552b73fd810f7dfa81a14849736291d84f91da2f8b5a58bcfaf0cc018962f4'),
             ('holdout.csv', '083821b17b5c7fdf81de46633ca08a7d601bd8266b269faa3911938d26bf66bf')]:
    assert hashlib.sha256(open(f, 'rb').read()).hexdigest() == h, f
df = pd.read_csv('data.csv')
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(hold_idx) == 1400 and len(set(hold_idx)) == 1400
train_idx = np.setdiff1d(np.arange(len(df)), hold_idx)
print(len(train_idx), len(hold_idx))

## Features
`device_serial` is excluded (identifier). `channel`, `device_age_years`, `price_eur`, `daily_hours` are all recorded at sale/registration, so they are known at prediction time.

In [ ]:
target = 'claim_within_year'
cat = ['channel']; num = ['device_age_years', 'price_eur', 'daily_hours']
features = cat + num
X = df[features]; y = df[target]
Xtr, ytr = X.iloc[train_idx], y.iloc[train_idx]
print(ytr.mean())
print(Xtr.describe())

## Candidates (named before any is scored)

In [ ]:
def pre(scale):
    return ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), cat),
                              ('n', StandardScaler() if scale else 'passthrough', num)])
candidates = {
    'baseline_prior': Pipeline([('p', pre(False)), ('m', DummyClassifier(strategy='prior'))]),
    'logreg_C1': Pipeline([('p', pre(True)), ('m', LogisticRegression(C=1.0, max_iter=1000))]),
    'logreg_C0.1': Pipeline([('p', pre(True)), ('m', LogisticRegression(C=0.1, max_iter=1000))]),
    'random_forest': Pipeline([('p', pre(False)), ('m', RandomForestClassifier(
        n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1))]),
    'hist_gb': Pipeline([('p', pre(False)), ('m', HistGradientBoostingClassifier(
        max_depth=3, learning_rate=0.05, max_iter=150, random_state=0))]),
}
print(list(candidates))

## Cross-validation on training rows
Preprocessing lives inside each pipeline, so it is fit per fold.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
splits = list(skf.split(Xtr, ytr))
cv_scores = {}
for name, est in candidates.items():
    cv_scores[name] = [
        roc_auc_score(ytr.iloc[va], clone(est).fit(Xtr.iloc[tr], ytr.iloc[tr]).predict_proba(Xtr.iloc[va])[:, 1])
        for tr, va in splits]
cv = pd.DataFrame({k: {'mean': np.mean(v), 'std': np.std(v)} for k, v in cv_scores.items()}).T
cv

## Choice (highest mean CV ROC AUC) and single holdout scoring

In [ ]:
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
print('chosen:', chosen)
final = clone(candidates[chosen]).fit(Xtr, ytr)
holdout_auc = roc_auc_score(y.iloc[hold_idx], final.predict_proba(X.iloc[hold_idx])[:, 1])
print('holdout roc_auc:', holdout_auc)

In [ ]:
# fold indices are positions within the training rows; map to data.csv row positions
cv_folds = [[[int(train_idx[i]) for i in tr], [int(train_idx[i]) for i in va]] for tr, va in splits]
evidence = {
    'features': features,
    'candidates': list(candidates),
    'cv_folds': cv_folds,
    'cv_scores': {k: [float(x) for x in v] for k, v in cv_scores.items()},
    'chosen': chosen,
    'holdout_rows': [int(i) for i in hold_idx],
    'holdout_scores': {chosen: float(holdout_auc)},
}
json.dump(evidence, open('evidence.json', 'w'))